# AI Waste Sorter

### Project Overview
In this project we try to build an AI model that can classify the kind of waste into 12 classes from images.

# Model Deployment Preparation

This Notebook will contain processes for:
1. Load and test MobileNetV2 model
2. Convert to TensorFlow.js format
3. Generate metadata for deployment
4. Create inference example code

In [1]:
import os
import numpy as np
import json
from pathlib import Path
import shutil

import tensorflow as tf
from tensorflow import keras
from PIL import Image

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

2026-02-08 17:53:46.666163: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770548026.776637   10321 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770548026.809043   10321 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770548027.076429   10321 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770548027.076463   10321 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770548027.076465   10321 computation_placer.cc:177] computation placer alr

TensorFlow version: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
BASE_DIR = Path.cwd()
MODELS_DIR = BASE_DIR / 'models'
DEPLOYMENT_DIR = BASE_DIR / 'deployment'
DEPLOYMENT_DIR.mkdir(exist_ok=True)

TFJS_DIR = DEPLOYMENT_DIR / 'tfjs_model'
TFJS_DIR.mkdir(exist_ok=True)

print(f"Models : {MODELS_DIR}")
print(f"Deployment directory: {DEPLOYMENT_DIR}")
print(f"TensorFlow.js output: {TFJS_DIR}")

Models : /mnt/d/personal/AI and Machine Learning/waste sorter ai/models
Deployment directory: /mnt/d/personal/AI and Machine Learning/waste sorter ai/deployment
TensorFlow.js output: /mnt/d/personal/AI and Machine Learning/waste sorter ai/deployment/tfjs_model


## 1. Load Best Model (MobileNetV2)

In [3]:
model_path = MODELS_DIR / 'mobilenetv2_final_best.keras'

print(f"Load model : {model_path}")
print(f"Model exists: {model_path.exists()}")

model = keras.models.load_model(model_path)
model.summary()

Load model : /mnt/d/personal/AI and Machine Learning/waste sorter ai/models/mobilenetv2_final_best.keras
Model exists: True


I0000 00:00:1770548060.897186   10321 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1766 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Model: "MobileNetV2_Transfer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 1280)           │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 12)             │         1,548 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,071,078 (26.97 MB)

 Trainable params: 2,224,588 (8.49 MB)

 Non-trainable params: 397,312 (1.52 MB)

 Optimizer params: 4,449,178 (16.97 MB)

In [4]:
with open(MODELS_DIR / 'mobilenetv2_results.json', 'r') as f:
    model_results = json.load(f)

print(f"Test Accuracy: {model_results['test_accuracy']:.4f}")
print(f"Parameters: {model_results['parameters']:,}")
print(f"Model Type: {model_results['model_type']}")

Test Accuracy: 0.9704
Parameters: 2,621,900
Model Type: Transfer Learning


## 2. Get Class Labels

In [5]:
PROCESSED_DIR = BASE_DIR / 'processed_data'
TRAIN_DIR = PROCESSED_DIR / 'train_balanced'

class_names = sorted([d.name for d in TRAIN_DIR.iterdir() if d.is_dir()])

print(f"Total classes: {len(class_names)}")
print(f"\nClass names:")
for i, name in enumerate(class_names):
    print(f"{i}: {name}")

Total classes: 12

Class names:
0: battery
1: biological
2: brown-glass
3: cardboard
4: clothes
5: green-glass
6: metal
7: paper
8: plastic
9: shoes
10: trash
11: white-glass


## 3. Test Model Inference

In [6]:
def preprocess_image(image_path, target_size=(224, 224)):
    """
    Preprocess image for inference
    """
    img = Image.open(image_path)
    if img.mode != 'RGB':
        img = img.convert('RGB')
    
    img = img.resize(target_size, Image.BILINEAR)
    img_array = np.array(img, dtype=np.float32)
    img_array = img_array / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    
    return img_array

def predict_waste_type(model, image_path, class_names):
    """
    Predict waste type from image
    """
    img_array = preprocess_image(image_path)
    predictions = model.predict(img_array, verbose=0)
    
    predicted_class_idx = np.argmax(predictions[0])
    confidence = predictions[0][predicted_class_idx]
    
    top_5_idx = np.argsort(predictions[0])[-5:][::-1]
    top_5_predictions = [
        {
            'class': class_names[idx],
            'confidence': float(predictions[0][idx])
        }
        for idx in top_5_idx
    ]
    
    return {
        'predicted_class': class_names[predicted_class_idx],
        'confidence': float(confidence),
        'top_5': top_5_predictions
    }

In [7]:
TEST_DIR = PROCESSED_DIR / 'test'
sample_category = class_names[0]
sample_images = list((TEST_DIR / sample_category).glob('*.jpg'))[:3]

print(f"Testing with {len(sample_images)} sample images from '{sample_category}':")
print()

for img_path in sample_images:
    result = predict_waste_type(model, img_path, class_names)
    print(f"Image: {img_path.name}")
    print(f"Predicted: {result['predicted_class']} (confidence: {result['confidence']:.4f})")
    print(f"Top 3 predictions:")
    for pred in result['top_5'][:3]:
        print(f"  - {pred['class']:15} : {pred['confidence']:.4f}")
    print()

Testing with 3 sample images from 'battery':



I0000 00:00:1770548064.808333   10486 service.cc:152] XLA service 0x77ee90002360 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1770548064.808380   10486 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 Laptop GPU, Compute Capability 8.6
2026-02-08 17:54:24.874091: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1770548065.387556   10486 cuda_dnn.cc:529] Loaded cuDNN version 91801
2026-02-08 17:54:33.867733: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng3{k11=2} for conv %cudnn-conv.91 = (f32[1,576,14,14]{3,2,1,0}, u8[0]{0}) custom-call(f32[1,576,14,14]{3,2,1,0} %bitcast.5750, f32[576,1,3,3]{3,2,1,0} %bitcast.5757), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, feature_group_count=576, custom_call_target="__cudnn$convForward", metadata={op_type="De

Image: battery12.jpg
Predicted: battery (confidence: 0.9998)
Top 3 predictions:
  - battery         : 0.9998
  - metal           : 0.0001
  - green-glass     : 0.0001

Image: battery127.jpg
Predicted: battery (confidence: 1.0000)
Top 3 predictions:
  - battery         : 1.0000
  - metal           : 0.0000
  - brown-glass     : 0.0000

Image: battery129.jpg
Predicted: battery (confidence: 1.0000)
Top 3 predictions:
  - battery         : 1.0000
  - cardboard       : 0.0000
  - paper           : 0.0000



## 4. Convert to TensorFlow.js Format

In [8]:
%pip install tensorflowjs


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
print("Convert model")
print(f"Input: {model_path}")
print(f"Output: {TFJS_DIR}")
print()

import tensorflowjs as tfjs

try:
    print("Load keras model")
    keras_model = keras.models.load_model(str(model_path))
    
    print("Saving to TensorFlow SavedModel format")
    saved_model_dir = DEPLOYMENT_DIR / 'temp_saved_model'
    saved_model_dir.mkdir(exist_ok=True)
    keras_model.export(str(saved_model_dir))
    
    print("Convert SavedModel to TensorFlow.js")
    tfjs.converters.convert_tf_saved_model(
        saved_model_dir=str(saved_model_dir),
        output_dir=str(TFJS_DIR),
        saved_model_tags='serve'
    )
    
    print("Cleaning up temporary files")
    import shutil
    shutil.rmtree(saved_model_dir)
    
    print("\nSuccess")
    
except Exception as e:
    print("\nFailed")
    print(f"Error: {str(e)}")
    import traceback
    traceback.print_exc()

Convert model
Input: /mnt/d/personal/AI and Machine Learning/waste sorter ai/models/mobilenetv2_final_best.keras
Output: /mnt/d/personal/AI and Machine Learning/waste sorter ai/deployment/tfjs_model



Load keras model
Saving to TensorFlow SavedModel format
INFO:tensorflow:Assets written to: /mnt/d/personal/AI and Machine Learning/waste sorter ai/deployment/temp_saved_model/assets


INFO:tensorflow:Assets written to: /mnt/d/personal/AI and Machine Learning/waste sorter ai/deployment/temp_saved_model/assets


Saved artifact at '/mnt/d/personal/AI and Machine Learning/waste sorter ai/deployment/temp_saved_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 12), dtype=tf.float32, name=None)
Captures:
  131871469980128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131871469990864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131871469992976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131871469986288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131871469988576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131871469984880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131871470017120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131871470020640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131871470016944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131871470019408

I0000 00:00:1770548099.927514   10321 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
I0000 00:00:1770548099.927743   10321 single_machine.cc:374] Starting new session
I0000 00:00:1770548099.928371   10321 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1766 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Cleaning up temporary files

Success


In [10]:
tfjs_files = list(TFJS_DIR.glob('*'))
print("\nTensorFlow.js model files:")
for file in tfjs_files:
    size_mb = file.stat().st_size / (1024 * 1024)
    print(f"  {file.name:40} - {size_mb:.2f} MB")

total_size = sum(f.stat().st_size for f in tfjs_files) / (1024 * 1024)
print(f"\nTotal model size: {total_size:.2f} MB")


TensorFlow.js model files:
  group1-shard1of3.bin                     - 4.00 MB
  group1-shard2of3.bin                     - 4.00 MB
  group1-shard3of3.bin                     - 1.83 MB
  model.json                               - 0.13 MB

Total model size: 9.96 MB


## 5. Generate Metadata and Config

In [11]:
metadata = {
    'model_info': {
        'name': 'MobileNetV2 Waste Classifier',
        'version': '1.0.0',
        'architecture': 'MobileNetV2',
        'type': 'Transfer Learning',
        'framework': 'TensorFlow/Keras',
        'converted_format': 'TensorFlow.js'
    },
    'performance': {
        'test_accuracy': model_results['test_accuracy'],
        'validation_accuracy': model_results['best_val_accuracy'],
        'parameters': model_results['parameters'],
        'model_size_mb': round(total_size, 2)
    },
    'input': {
        'shape': [224, 224, 3],
        'dtype': 'float32',
        'preprocessing': {
            'resize': [224, 224],
            'normalize': 'divide_by_255',
            'color_mode': 'RGB'
        }
    },
    'output': {
        'shape': [12],
        'dtype': 'float32',
        'type': 'categorical',
        'activation': 'softmax'
    },
    'classes': {
        'num_classes': len(class_names),
        'labels': class_names,
        'mapping': {i: name for i, name in enumerate(class_names)}
    },
    'training': {
        'dataset': 'Garbage Classification',
        'total_samples': 15515,
        'train_samples': 8424,
        'val_samples': 1550,
        'test_samples': 1556,
        'epochs': model_results['epochs_trained'],
        'batch_size': 32
    }
}

with open(DEPLOYMENT_DIR / 'model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("\nMetadata preview:")
print(json.dumps(metadata, indent=2))


Metadata preview:
{
  "model_info": {
    "name": "MobileNetV2 Waste Classifier",
    "version": "1.0.0",
    "architecture": "MobileNetV2",
    "type": "Transfer Learning",
    "framework": "TensorFlow/Keras",
    "converted_format": "TensorFlow.js"
  },
  "performance": {
    "test_accuracy": 0.9704370179948586,
    "validation_accuracy": 0.9651612639427185,
    "parameters": 2621900,
    "model_size_mb": 9.96
  },
  "input": {
    "shape": [
      224,
      224,
      3
    ],
    "dtype": "float32",
    "preprocessing": {
      "resize": [
        224,
        224
      ],
      "normalize": "divide_by_255",
      "color_mode": "RGB"
    }
  },
  "output": {
    "shape": [
      12
    ],
    "dtype": "float32",
    "type": "categorical",
    "activation": "softmax"
  },
  "classes": {
    "num_classes": 12,
    "labels": [
      "battery",
      "biological",
      "brown-glass",
      "cardboard",
      "clothes",
      "green-glass",
      "metal",
      "paper",
      "plasti

## 6. Generate JavaScript Inference Code

In [12]:
js_inference_code = '''// Waste Classification Model - Inference Code
// MobileNetV2

import * as tf from '@tensorflow/tfjs';

// Model configuration
const MODEL_URL = './tfjs_model/model.json';
const IMAGE_SIZE = 224;
const CLASS_NAMES = ''' + json.dumps(class_names, indent=2) + ''';

// Load model
let model = null;

async function loadModel() {
  console.log('Load model');
  model = await tf.loadGraphModel(MODEL_URL);
  console.log('Model loaded successfully');
  return model;
}

// Preprocess image
function preprocessImage(imageElement) {
  return tf.tidy(() => {
    // Convert image to tensor
    let tensor = tf.browser.fromPixels(imageElement)
      .resizeBilinear([IMAGE_SIZE, IMAGE_SIZE])
      .toFloat();
    
    // Normalize to [0, 1]
    tensor = tensor.div(255.0);
    
    // Add batch dimension
    tensor = tensor.expandDims(0);
    
    return tensor;
  });
}

// Predict waste type
async function predictWasteType(imageElement) {
  if (!model) {
    throw new Error('Model not loaded. Call loadModel() first.');
  }
  
  // Preprocess image
  const preprocessed = preprocessImage(imageElement);
  
  // Make prediction
  const predictions = await model.predict(preprocessed);
  const probabilities = await predictions.data();
  
  // Get top prediction
  const maxIdx = probabilities.indexOf(Math.max(...probabilities));
  const predictedClass = CLASS_NAMES[maxIdx];
  const confidence = probabilities[maxIdx];
  
  // Get top 5 predictions
  const top5 = Array.from(probabilities)
    .map((prob, idx) => ({ class: CLASS_NAMES[idx], confidence: prob }))
    .sort((a, b) => b.confidence - a.confidence)
    .slice(0, 5);
  
  // Cleanup
  preprocessed.dispose();
  predictions.dispose();
  
  return {
    predictedClass,
    confidence,
    top5
  };
}

// Example usage
async function main() {
  // Load model
  await loadModel();
  
  // Get image element (from file input, webcam, etc.)
  const imageElement = document.getElementById('input-image');
  
  // Predict
  const result = await predictWasteType(imageElement);
  
  console.log('Prediction:', result.predictedClass);
  console.log('Confidence:', (result.confidence * 100).toFixed(2) + '%');
  console.log('Top 5:', result.top5);
}

export { loadModel, predictWasteType };
'''

with open(DEPLOYMENT_DIR / 'inference.js', 'w') as f:
    f.write(js_inference_code)

## 7. Generate React Component Example

In [13]:
react_component = '''import React, { useState, useEffect, useRef } from 'react';
import * as tf from '@tensorflow/tfjs';

const WasteClassifier = () => {
  const [model, setModel] = useState(null);
  const [loading, setLoading] = useState(true);
  const [prediction, setPrediction] = useState(null);
  const [imagePreview, setImagePreview] = useState(null);
  const imageRef = useRef(null);
  const fileInputRef = useRef(null);

  const CLASS_NAMES = ''' + json.dumps(class_names, indent=2) + ''';

  // Load model on component mount
  useEffect(() => {
    loadModel();
  }, []);

  const loadModel = async () => {
    try {
      setLoading(true);
      const loadedModel = await tf.loadGraphModel('/tfjs_model/model.json');
      setModel(loadedModel);
      setLoading(false);
      console.log('Model loaded successfully!');
    } catch (error) {
      console.error('Error loading model:', error);
      setLoading(false);
    }
  };

  const preprocessImage = (imageElement) => {
    return tf.tidy(() => {
      let tensor = tf.browser.fromPixels(imageElement)
        .resizeBilinear([224, 224])
        .toFloat()
        .div(255.0)
        .expandDims(0);
      return tensor;
    });
  };

  const handleImageUpload = (event) => {
    const file = event.target.files[0];
    if (file) {
      const reader = new FileReader();
      reader.onload = (e) => {
        setImagePreview(e.target.result);
      };
      reader.readAsDataURL(file);
    }
  };

  const classifyImage = async () => {
    if (!model || !imageRef.current) return;

    try {
      const preprocessed = preprocessImage(imageRef.current);
      const predictions = await model.predict(preprocessed);
      const probabilities = await predictions.data();

      const maxIdx = probabilities.indexOf(Math.max(...probabilities));
      const predictedClass = CLASS_NAMES[maxIdx];
      const confidence = probabilities[maxIdx];

      const top5 = Array.from(probabilities)
        .map((prob, idx) => ({ class: CLASS_NAMES[idx], confidence: prob }))
        .sort((a, b) => b.confidence - a.confidence)
        .slice(0, 5);

      setPrediction({
        predictedClass,
        confidence,
        top5
      });

      preprocessed.dispose();
      predictions.dispose();
    } catch (error) {
      console.error('Error during prediction:', error);
    }
  };

  return (
    <div className="waste-classifier">
      <h1>AI Waste Sorter</h1>
      <p>MobileNetV2 - 97% Accuracy</p>

      {loading && <p>Loading model...</p>}

      <div className="upload-section">
        <input
          ref={fileInputRef}
          type="file"
          accept="image/*"
          onChange={handleImageUpload}
          style={{ display: 'none' }}
        />
        <button onClick={() => fileInputRef.current.click()}>
          Upload Image
        </button>
      </div>

      {imagePreview && (
        <div className="preview-section">
          <img
            ref={imageRef}
            src={imagePreview}
            alt="Preview"
            style={{ maxWidth: '400px' }}
            onLoad={classifyImage}
          />
        </div>
      )}

      {prediction && (
        <div className="results">
          <h2>Classification Result</h2>
          <p className="prediction">
            <strong>{prediction.predictedClass}</strong>
          </p>
          <p className="confidence">
            Confidence: {(prediction.confidence * 100).toFixed(2)}%
          </p>
          <h3>Top 5 Predictions:</h3>
          <ul>
            {prediction.top5.map((item, idx) => (
              <li key={idx}>
                {item.class}: {(item.confidence * 100).toFixed(2)}%
              </li>
            ))}
          </ul>
        </div>
      )}
    </div>
  );
};

export default WasteClassifier;
'''

with open(DEPLOYMENT_DIR / 'WasteClassifier.jsx', 'w') as f:
    f.write(react_component)

## 8. Generate README for Deployment

In [14]:
readme_content = f'''# AI Waste Sorter - Deployment Guide

## Model Information

- **Model**: MobileNetV2 Transfer Learning
- **Test Accuracy**: {model_results['test_accuracy']*100:.2f}%
- **Model Size**: {total_size:.2f} MB
- **Parameters**: {model_results['parameters']:,}
- **Input Size**: 224x224x3 (RGB)
- **Output Classes**: {len(class_names)}

## Classes

{chr(10).join([f'{i}. {name}' for i, name in enumerate(class_names)])}

## Quick Start

### 1. Install Dependencies

```bash
npm install @tensorflow/tfjs
```

### 2. Copy Model Files

Copy the `tfjs_model` folder to your React project's `public` directory:

```
public/
  tfjs_model/
    model.json
    group1-shard*.bin
```

### 3. Use in React

```javascript
import WasteClassifier from './WasteClassifier';

function App() {{
  return (
    <div className="App">
      <WasteClassifier />
    </div>
  );
}}
```

## Image Preprocessing

Before prediction, images must be:
1. Resized to 224x224 pixels
2. Converted to RGB (3 channels)
3. Normalized to [0, 1] range (divide by 255)
4. Batch dimension added (shape: [1, 224, 224, 3])

## Example Code

See `inference.js` for vanilla JavaScript implementation.
See `WasteClassifier.jsx` for React component implementation.

## Performance

- **Browser Inference Time**: ~50-100ms (depends on device)
- **Model Load Time**: ~1-2 seconds
- **Memory Usage**: ~50-100 MB

## Supported Browsers

- Chrome 57+
- Firefox 52+
- Safari 11+
- Edge 79+

## Notes

- Model works best with clear, well-lit images
- Single waste item per image recommended
- Confidence threshold: Use predictions with >70% confidence
- Model trained on balanced dataset (702 images per class)

## Files Included

- `tfjs_model/` - TensorFlow.js model files
- `model_metadata.json` - Model information and configuration
- `inference.js` - Vanilla JavaScript inference code
- `WasteClassifier.jsx` - React component example
- `README.md` - This file

## Support

For issues or questions, please refer to:
- TensorFlow.js documentation: https://www.tensorflow.org/js
- Model training notebook: `04a_train_mobilenetv2.ipynb`
'''

with open(DEPLOYMENT_DIR / 'README.md', 'w') as f:
    f.write(readme_content)


## 9. Summary

In [15]:
deployment_files = list(DEPLOYMENT_DIR.rglob('*'))
deployment_files = [f for f in deployment_files if f.is_file()]

print("DEPLOYMENT PREPARATION COMPLETE")
print("\nGenerated files:")
for file in sorted(deployment_files):
    rel_path = file.relative_to(DEPLOYMENT_DIR)
    size_kb = file.stat().st_size / 1024
    print(f"  {str(rel_path):50} - {size_kb:>8.2f} KB")

print(f"\nTotal files: {len(deployment_files)}")
total_deploy_size = sum(f.stat().st_size for f in deployment_files) / (1024 * 1024)
print(f"Total deployment size: {total_deploy_size:.2f} MB")

print("\nModel Performance:")
print(f"  Test Accuracy: {model_results['test_accuracy']*100:.2f}%")
print(f"  Model Size: {total_size:.2f} MB")
print(f"  Classes: {len(class_names)}")

DEPLOYMENT PREPARATION COMPLETE

Generated files:
  README.md                                          -     2.02 KB
  WasteClassifier.jsx                                -     3.80 KB
  inference.js                                       -     2.31 KB
  model_metadata.json                                -     1.51 KB
  tfjs_model/group1-shard1of3.bin                    -  4096.00 KB
  tfjs_model/group1-shard2of3.bin                    -  4096.00 KB
  tfjs_model/group1-shard3of3.bin                    -  1874.93 KB
  tfjs_model/model.json                              -   134.40 KB

Total files: 8
Total deployment size: 9.97 MB

Model Performance:
  Test Accuracy: 97.04%
  Model Size: 9.96 MB
  Classes: 12
